# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Structure-Preserving Neural Networks: Hamiltonian, Port-Hamiltonian, GENERIC, and Divergence-Free Fields
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Verlet, L. (1967). *Computer "Experiments" on Classical Fluids. I. Thermodynamical Properties of Lennard-Jones Molecules*. Physical Review.
- Greydanus, S., Dzamba, M., & Yosinski, J. (2019). *Hamiltonian Neural Networks*. NeurIPS.
- Hernández, Q., Bermejo, I., & Chinesta, F. (2021). *Structure-Preserving Neural Networks*. Journal of Computational Physics.
- Desai, S., Mattheakis, M., & Joy, H. (2021). *Port-Hamiltonian Neural Networks for Learning Open Physical Systems*. Physical Review E.

## 2. Mathematical Foundations & Governing Equations

### Hamiltonian Neural Networks (HNN)
$$\dot{z} = J \nabla \mathcal{H}(z) = \begin{bmatrix} \dot{q} \\ \dot{p} \end{bmatrix} = \begin{bmatrix} \nabla_p \mathcal{H} \\ -\nabla_q \mathcal{H} \end{bmatrix}, \quad \frac{d\mathcal{H}}{dt} = (\nabla \mathcal{H})^T J \nabla \mathcal{H} \equiv 0$$

### Port-Hamiltonian Dynamics with Dissipation
$$\dot{z} = (J - R)\nabla \mathcal{H}(z), \quad R \succeq 0 \implies \frac{d\mathcal{H}}{dt} \le 0$$

### Thermodynamic GENERIC Framework
$$\dot{z} = L \nabla E + M \nabla S \quad \text{s.t.} \quad L \nabla S = 0, \quad M \nabla E = 0 \implies \frac{dE}{dt} = 0, \quad \frac{dS}{dt} \ge 0$$

### Divergence-Free Solenoidal Vector Fields
$$\mathbf{v}(x, y) = \begin{bmatrix} \partial_y \psi \\ -\partial_x \psi \end{bmatrix} \implies \nabla \cdot \mathbf{v} = \frac{\partial^2 \psi}{\partial x \partial y} - \frac{\partial^2 \psi}{\partial y \partial x} \equiv 0$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
from safetensors.torch import load_file

torch.manual_seed(42)

# Phase state: B=4, q in R^2, p in R^2 -> z in R^4
z_state = torch.randn(4, 4)
q, p = z_state[:, :2], z_state[:, 2:]

# Potential V(q) parameters
w_v1 = torch.randn(2, 8) * 0.5
b_v1 = torch.randn(8) * 0.1
w_v2 = torch.randn(8, 1) * 0.5
b_v2 = torch.randn(1) * 0.1

print(f"Phase state: {z_state.shape}, q: {q.shape}, p: {p.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def compute_v_and_grad(q_in, w1, b1, w2, b2):
    z = q_in @ w1 + b1
    a = torch.tanh(z)
    v = a @ w2 + b2
    temp = (1.0 - a**2) * w2.t()
    grad_v = temp @ w1.t()
    return v, grad_v

v_potential, grad_v = compute_v_and_grad(q, w_v1, b_v1, w_v2, b_v2)
t_kinetic = 0.5 * torch.sum(p**2, dim=-1, keepdim=True)
h_total = t_kinetic + v_potential

# 1. HNN Dynamics
z_dot_hnn = torch.cat([p, -grad_v], dim=-1)

# 2. Port-Hamiltonian with dissipation gamma = 0.2
gamma = 0.2
z_dot_phnn = torch.cat([p, -grad_v - gamma * p], dim=-1)

# 3. Symplectic Leapfrog Integration step with h = 0.1
h = 0.1
p_half = p - 0.5 * h * grad_v
q_next = q + h * p_half
_, grad_v_next = compute_v_and_grad(q_next, w_v1, b_v1, w_v2, b_v2)
p_next = p_half - 0.5 * h * grad_v_next
z_leapfrog = torch.cat([q_next, p_next], dim=-1)

# 4. Divergence-free solenoidal field
w_psi1 = torch.randn(2, 8) * 0.5
b_psi1 = torch.randn(8) * 0.1
w_psi2 = torch.randn(8, 1) * 0.5
b_psi2 = torch.randn(1) * 0.1
psi_stream, grad_psi = compute_v_and_grad(q, w_psi1, b_psi1, w_psi2, b_psi2)
v_solenoidal = torch.stack([grad_psi[:, 1], -grad_psi[:, 0]], dim=-1)

print(f"HNN dot: {z_dot_hnn.shape}, Leapfrog: {z_leapfrog.shape}, Solenoidal: {v_solenoidal.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Exact energy conservation in HNN (dH/dt = 0)
dH_dt = torch.sum(grad_v * z_dot_hnn[:, :2] + p * z_dot_hnn[:, 2:], dim=-1)
assert torch.allclose(dH_dt, torch.zeros_like(dH_dt), atol=1e-5), "HNN energy not conserved"

# Invariant 2: Dissipation in Port-Hamiltonian (dH/dt <= 0)
dH_dt_phnn = torch.sum(grad_v * z_dot_phnn[:, :2] + p * z_dot_phnn[:, 2:], dim=-1)
assert torch.all(dH_dt_phnn <= 1e-6), "Port-Hamiltonian energy increased"

# Invariant 3: Solenoidal velocity divergence is zero
xy_grad = q.clone().detach().requires_grad_(True)
_, g_psi = compute_v_and_grad(xy_grad, w_psi1, b_psi1, w_psi2, b_psi2)
v_x = g_psi[:, 1:2]
v_y = -g_psi[:, 0:1]
div_x = torch.autograd.grad(v_x, xy_grad, grad_outputs=torch.ones_like(v_x), create_graph=True)[0][:, 0:1]
div_y = torch.autograd.grad(v_y, xy_grad, grad_outputs=torch.ones_like(v_y), create_graph=True)[0][:, 1:2]
div_v = div_x + div_y
assert torch.allclose(div_v, torch.zeros_like(div_v), atol=1e-5), "Solenoidal divergence non-zero"
print("All mathematical invariants (Hamiltonian conservation, passivity, solenoidal divergence-free) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = compute_v_and_grad(q, w_v1, b_v1, w_v2, b_v2)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Symplectic gradient evaluation step latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Explicit Forward Euler integration on Hamiltonian system causes energy explosion
q_euler = q.clone()
p_euler = p.clone()
for _ in range(20):
    _, g_v = compute_v_and_grad(q_euler, w_v1, b_v1, w_v2, b_v2)
    q_euler = q_euler + h * p_euler
    p_euler = p_euler - h * g_v
v_euler, _ = compute_v_and_grad(q_euler, w_v1, b_v1, w_v2, b_v2)
h_euler = 0.5 * torch.sum(p_euler**2, dim=-1, keepdim=True) + v_euler
energy_drift_euler = torch.mean(h_euler - h_total).item()
print(f"Pathological Forward Euler 20-step energy growth: {energy_drift_euler:.4f}")

# Corrected: Symplectic Leapfrog preserves energy bounded in an oscillation band
q_lf = q.clone()
p_lf = p.clone()
for _ in range(20):
    _, g_v0 = compute_v_and_grad(q_lf, w_v1, b_v1, w_v2, b_v2)
    p_h = p_lf - 0.5 * h * g_v0
    q_lf = q_lf + h * p_h
    _, g_v1 = compute_v_and_grad(q_lf, w_v1, b_v1, w_v2, b_v2)
    p_lf = p_h - 0.5 * h * g_v1
v_lf, _ = compute_v_and_grad(q_lf, w_v1, b_v1, w_v2, b_v2)
h_lf = 0.5 * torch.sum(p_lf**2, dim=-1, keepdim=True) + v_lf
energy_drift_lf = torch.mean(torch.abs(h_lf - h_total)).item()
print(f"Corrected Symplectic Leapfrog bounded drift: {energy_drift_lf:.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Exact Symplectic Invariance**: Hamiltonian neural networks preserve phase space volume and total energy by construction through skew-symmetric gradients.
2. **Controlled Dissipation**: Port-Hamiltonian networks provide guaranteed passivity and Lyapunov stability for open robotics control.
3. **Exact Solenoidal Fluid Mechanics**: Parameterizing vector fields via stream/curl potentials enforces zero divergence ($\nabla \cdot \mathbf{v} \equiv 0$) identically without soft penalties.